In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.')))
import torch
import pandas as pd
from modules.preprocessing import load_config, load_and_split, OUTPUT_NAMES, INPUT_COLS
from modules.training import run_iterative_training, plot_results, save_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET = 'Vy'

In [ ]:
YAML_PATH = Path('hyperparameters.yaml')
hp, DS, DNN_DEFAULTS, OUTS, TRAIN_SPLIT, R2_THRESHOLD, INCREMENT, MAX_ITER = load_config(YAML_PATH)
print(f'Target: {TARGET}  |  R2_threshold={R2_THRESHOLD}  increment={INCREMENT}  max_iter={MAX_ITER}')

In [ ]:
DATA_PATH = Path('../1. Dataset/DNN_dataset_cleaned.csv')
X_train_all, X_test, Y_train_all, Y_test, scaler = load_and_split(
    DATA_PATH, INPUT_COLS, OUTPUT_NAMES, TRAIN_SPLIT
)
print(f'X_train_all: {X_train_all.shape}  |  X_test: {X_test.shape}')

In [ ]:
result = run_iterative_training(
    TARGET, X_train_all, Y_train_all, X_test, Y_test,
    INCREMENT, MAX_ITER, R2_THRESHOLD,
    OUTPUT_NAMES, OUTS, DNN_DEFAULTS, device
)

In [ ]:
print(f"\nBest model for {TARGET}")
print(f"  Dataset size : {result['final_n']:,}")
print(f"  Train R2     : {result['train_r2']:.4f}")
print(f"  Test R2      : {result['test_r2']:.4f}")
print(f"  R2 >= 0.90   : {'YES' if result['test_r2'] >= R2_THRESHOLD else 'NO'}")

In [ ]:
plot_results(TARGET, result, R2_THRESHOLD, save_path=f'results_{TARGET}.png')

In [ ]:
save_model(result['model'], TARGET, Path('saved_models'))